In [ ]:
# make local editable packages automatically reload
%load_ext autoreload
%autoreload 2

In [ ]:
# Import dependencies
import numpy as np
from omnipose import models, core

# This checks to see if you have set up your GPU properly.
# CPU performance is a lot slower, but not a problem if you 
# are only processing a few images.
use_GPU = core.use_gpu()
print('>>> GPU activated? {}'.format(use_GPU))

# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.dpi'] = 300
plt.style.use('dark_background')
%matplotlib inline
import omnipose, omnipose
from omnipose.plot import imshow


In [ ]:
from omnipose import io, transforms, plot, models, core

import omnipose
import skimage.io
from omnipose import models
import omnipose
use_GPU = omnipose.core.use_gpu()
# use_GPU = 0

# now use 3D model
modeldir = '/home/kcutler/DataDrive/3D_spacetime/linked/models/cellpose_residual_on_style_on_concatenation_off_omni_abstract_nclasses_2_nchan_1_dim_3_linked_2023_12_29_13_34_19.101627'

#earier - oope, deleted these 
# modeldir = '/Volumes/DataDrive/3D_spacetime/linked/models/cellpose_residual_on_style_on_concatenation_off_omni_abstract_nclasses_2_nchan_1_dim_3_linked_2023_11_29_03_29_32.411384'
# modeldir = '/Volumes/DataDrive/3D_spacetime/linked/models/cellpose_residual_on_style_on_concatenation_off_omni_abstract_nclasses_2_nchan_1_dim_3_linked_2023_12_10_00_55_54.580168'

# modeldir = '/Volumes/DataDrive/3D_spacetime/linked/models/cellpose_residual_on_style_on_concatenation_off_omni_abstract_nclasses_2_nchan_1_dim_3_linked_2023_12_29_13_34_19.101627_epoch_7999'
modeldir = omnipose.io.adjust_file_path(modeldir)

nclasses = 2
dim = 3
model =  models.OmniModel(gpu=use_GPU, pretrained_model=modeldir, net_avg=False, diam_mean=0, nclasses=nclasses, dim=dim)
chans = None 

In [ ]:
from pathlib import Path
import os, re
file = '/Volumes/DataDrive/a_baylii_good_dnaN/xy17/2.tif'
file = '/Volumes/DataDrive/a_baylii_G_S/G2 cropped [0].tif'
n = 1
file = f'/Volumes/DataDrive/a_baylii_G_S/G2 cropped [{n}].tif'
file = '/Volumes/DataDrive/a_baylii_G_S/S3 cropped [9].tif'
file = '/Volumes/HiprDrive/kevin_high_frame_rate/ftsZ/cropped_processed/2_3_phase_align.tif'
file = '/Volumes/HiprDrive/kevin_high_frame_rate/ftsZ/2/2 XYPos:2 cropped [3].tif'
file = omnipose.io.adjust_file_path(file)
stack = omnipose.io.imread(file)
stack.shape

In [ ]:
ret = omnipose.stacks.cross_reg(stack[:,0], # in this data, channels is at index 1
                                   # reverse=1,
                                   max_shift=200, 
                                   normalization='phase', 
                                   # localnorm=0,
                                   # moving_reference=1,
                                   # upsample_factor=100
                                  ) 
# if moving_reference:
    
#     shifts, phase_shift = ret
# else:
# shifts = ret
shifts = ret.copy()

phase_shift = omnipose.stacks.shift_stack(stack[:,0],shifts, mode='reflect')

# fluor_shift = omnipose.stacks.shift_stack(stack[:,1],shifts)

slc = omnipose.stacks.shifts_to_slice(shifts,phase_shift.shape[-2:], pad=20)

phase = phase_shift[(Ellipsis,)+slc]
fluor = fluor_shift[(Ellipsis,)+slc]
# io.imwrite(os.path.join(savedir,name+'_img_crop_rev.tif'),phase)
# io.imwrite(os.path.join(savedir,name+'_img_crop_nonorm.tif'),phase)

print(phase.shape)
# imshow([phase_shift[0],phase_shift[-1]])
imshow([phase[i] for i in [0,1,2,3,4,5,6,7,8,9,-50, -20,-1]])

In [ ]:
imgs = phase
eval_set = omnipose.data.eval_set(phase[np.newaxis], # must be nbatch,TYX, mabe should make a catch here 
                                  dim=dim, 
                                  device=model.device,
                                  # normalize_stack=False,
                                  tile=True,
                                  rescale=1.0,
                                  extra_pad = 4,
                                  pad_mode='replicate'
                                 )

In [ ]:


nimg = len(imgs)
n = range(nimg)

params = {'channels':chans,
          # 'diameter': 0, 
          'rescale': 1.,
          'indices':[0],
          'show_progress': 0,
          'mask_threshold': -1,
          'num_workers': 0,
          'transparency': True,
          'flow_threshold': 0,
          'omni': True,
          'cluster': 1,
          # 'suppress':True,
          'resample': True,
          'verbose':0,
          'affinity_seg': 1,
          'tile': 1,
          'bsize':64, # side length of tile 
          'batch_size':4, # how many tiles to run simultaneously 
          'tile_overlap': .1,
          'min_size': 9,
          # 'compute_masks':0,
          # 'niter': None 
          'niter':10,
          'hysteresis':True,
        }

import torch
torch.cuda.empty_cache()
masks, flows, _ = model.eval(eval_set,**params)
# masks, flows, _ = model.eval(phase[np.newaxis],**params)

In [ ]:
import ocdkit.array
from omnipose import plot
import omnipose

k = 0
# for idx,i in enumerate(n):
T = [0,10,32,-1]

# T=range(120)
for idx,i in zip([t for t in T],[n[t] for t in T]):

    maski = masks[k][idx]
    flowi = flows[k][0][idx]
    
    szX = maski.shape[-1]
    dpi = min(mpl.rcParams['figure.dpi'], 100000 / szX)
    
    f = 10
    fig = plt.figure(figsize=(f,f*4),dpi=dpi)
    fig.patch.set_facecolor([0]*4)
    

    bdi = flows[k][-1][idx]
    plot.show_segmentation(fig, ocdkit.array.normalize99(imgs[i]), 
                           maski, flowi, bdi, channels=chans, omni=True, 
                           bg_color=0, interpolation='none')

    plt.tight_layout()
    plt.show()

In [ ]:
k = 0
aa = flows[k][6]
msk = masks[k]
lbl,logs = omnipose.core.split_spacetime(aa,msk)
rgb = omnipose.plot.rgb_flow(flows[k][1][-2:].swapaxes(0,1),transparency=True).cpu().numpy()

In [ ]:
imshow([omnipose.plot.apply_ncolor(lbl[-1]),omnipose.plot.apply_ncolor(logs[-1])],1)


In [ ]:
from pathlib import Path

name = omnipose.utils.getname(file)
savedir = Path(file).parent
savedir

In [ ]:
from omnipose import io
io.imwrite(file[:-4]+'_space_labels.tif',lbl)


io.imwrite(file[:-4]+'_logs.tif',logs)

io.imwrite(os.path.join(savedir,name+'_masks.tif'),masks[k])
io.imwrite(os.path.join(savedir,name+'_bd.tif'),flows[k][-1])
io.imwrite(os.path.join(savedir,name+'_rgb_flow.tif'),rgb)
io.imwrite(os.path.join(savedir,name+'_flow_t.tif'),flows[k][1][0])

In [ ]:
io.imwrite(os.path.join(savedir,name+'_img_crop.tif'),imgs)
io.imwrite(os.path.join(savedir,name+'_fluor_crop.tif'),fluor)


print(savedir)

### Segment frame by frame instead 

In [ ]:
%autoreload 2
del models
from omnipose import models

# modeldir = os.path.join(datadir,'omnipose_link_train/combined/models/cellpose_residual_on_style_on_concatenation_off_omni_nclasses_3_nchan_1_combined_2023_06_15_14_22_44.019995_epoch_3000')
# modeldir = os.path.join(datadir,'omnipose_link_train/combined/models/cellpose_residual_on_style_on_concatenation_off_omni_nclasses_3_nchan_1_combined_2023_06_15_14_22_44.019995_epoch_11999')
# modeldir = os.path.join(datadir,'omnipose_link_train/combined/models/cellpose_residual_on_style_on_concatenation_off_omni_abstract_nclasses_2_nchan_1_dim_2_combined_2023_09_27_14_03_28.868599_epoch_3999')

modeldir = '/home/kcutler/DataDrive/omnipose_link_train/combined/models/cellpose_residual_on_style_on_concatenation_off_omni_abstract_nclasses_2_nchan_1_dim_2_combined_2023_12_18_15_10_20.498521_epoch_3999'

modeldir = omnipose.io.adjust_file_path(modeldir)

params['channels']  = None
params['affinity_seg'] = True
params['rescale'] = 1
params['niter'] = None
params['tile'] = False


model_fbf = models.OmniModel(gpu=use_GPU, pretrained_model=modeldir, nclasses=2, nchan=1)

eval_set_fbf = omnipose.data.eval_set(imgs[0],dim=2,device=model.device)

# masks_fbf,flows_fbf, _ = model_fbf.eval(eval_set_fbf,**params)
masks_fbf,flows_fbf, _ = model_fbf.eval(imgs,**params)

In [ ]:
io.imwrite(os.path.join(savedir,name+'_masks_fbf.tif'),omnipose.stacks.make_unique(np.stack(masks_fbf)))

In [ ]:
imshow(masks_fbf[-10])

In [ ]:
savedir

In [ ]:
import bactrack
# !pip install mip

In [ ]:
from bactrack.tracking import OverlapWeight, IOUWeight, DistanceWeight
hier_arr = bactrack.widget.get_hierarchies_from_masks(masks_fbf)
w = OverlapWeight(hier_arr)

In [ ]:
from bactrack.tracking import MIPSolver, ScipySolver
solver = ScipySolver(w.weight_matrix, hier_arr)
n, e = solver.solve()

In [ ]:
masks, edge_df = bactrack.io.format_output(hier_arr, n, e, label_format = "kevin")
node_df = bactrack.io.hiers_to_df(hier_arr)

In [ ]:
import pandas as pd
hier_df = bactrack.io.hiers_to_df(hier_arr)
merged_df = pd.merge(edge_df, hier_df.add_suffix('_source'), left_on='Source Index', right_on='index_source', how='left')
merged_df = pd.merge(merged_df, hier_df.add_suffix('_target'), left_on='Target Index', right_on='index_target', how='left')
merged_df

In [ ]:
filtered_df = merged_df[["label_source", "label_target"]].astype(int)
result = filtered_df[filtered_df["label_source"] != filtered_df["label_target"]] 
result.head(10)

In [ ]:
merged_df["label_source"]

In [ ]:

import pandas as pd
import csv

output_file = os.path.join(savedir,name+'_links.txt') # change this
result.to_csv(output_file, header=False, index=False, sep=',', quoting=csv.QUOTE_NONE)

omnipose.io.imwrite(os.path.join(savedir,name+'_sherry_masks.tif'),masks)


In [ ]:
import napari
from napari.utils.notebook_display import nbscreenshot
import numpy as np

masks_int = [mask.astype(np.uint32) for mask in masks]
masks_3d = np.stack(masks_int, axis=0)


viewer = napari.Viewer()
viewer.window.resize(1800, 1000)
layers = viewer.add_image(phase, name="phase")
layers = viewer.add_labels(masks_3d, name="mask")
layers = viewer.add_labels(masks_3d_nc, name="ncolor")


viewer.dims.ndisplay = 3
viewer.camera.angles=(-180,0,45)
display(nbscreenshot(viewer))

In [ ]:
import ncolor
masks_3d_nc = ncolor.label(masks_3d)


In [ ]:
name

In [ ]:
links = omnipose.io.load_links(os.path.join(savedir,name+'_links.txt'))
masks = omnipose.io.imread(os.path.join(savedir,name+'_sherry_masks_edited.tif'))
# masks = omnipose.io.imread(os.path.join(savedir,name+'_sherry_masks.tif'))



In [ ]:
flows = omnipose.core.labels_to_flows([masks],[links], dim=3)
flows[0][1].shape

omnipose.io.imwrite(os.path.join(savedir,name+'_computed_dists.tif'),flows[0][-1])

In [ ]:
# flows[0][1].shape
masks.dtype